# Therapeutic Optimization — Colab Runner

This notebook is intentionally lightweight. All reusable logic lives in `src/therapeutic_optimization/`.

Pipeline: **T1 → UP1 → T2 → S1 → R1 → UB2 → R2**.

Before publishing, replace the placeholder GitHub URL below with the real repository URL.


## 0. Runtime

Use a **GPU runtime**. EUP uses ESM2-3B and this implementation intentionally fails rather than silently falling back to a very slow CPU path.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/YOUR_USERNAME/therapeutic_optimization.git"
REPO_DIR = Path("/content/therapeutic_optimization")

if "YOUR_USERNAME" in REPO_URL:
    print("Replace REPO_URL with your public GitHub repository after you upload it.")
else:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    print("Repository ready:", REPO_DIR)


## 1. Install dependencies

EUP assets and the ESM2 cache stay in the Colab runtime, not in your Drive/repository. ColabFold is installed separately because its JAX/CUDA packages are runtime-specific.


In [ ]:
# Run this after REPO_URL is configured and the repository has been cloned.
if REPO_DIR.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[eup]"], check=True)

    if shutil.which("git-lfs") is None:
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "git-lfs"], check=True)
    subprocess.run(["git", "lfs", "install"], check=True)

    if shutil.which("colabfold_batch") is None:
        print("colabfold_batch not found. Installing the current CUDA 12 ColabFold stack...")
        subprocess.run([
            sys.executable, "-m", "pip", "install", "-q",
            "colabfold[alphafold,openmm]", "jax[cuda12]", "openmm[cuda12]"
        ], check=True)

    subprocess.run([sys.executable, str(REPO_DIR / "scripts" / "check_environment.py")], check=False)


## 2. Choose where run outputs live

Set `SAVE_TO_DRIVE = True` if you want the run to survive Colab shutdown. The package code still runs from GitHub; only `storage/` outputs go into this run directory.


In [ ]:
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_ROOT = Path("/content/drive/MyDrive/therapeutic_optimization_run")
else:
    RUN_ROOT = Path("/content/therapeutic_optimization_run")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Run root:", RUN_ROOT)


## 3. Hyperparameters

`single` changes one selected lysine at a time. `combinatorial` generates orders 1 through `MAX_COMBINATION_ORDER` using every replacement amino acid listed in `REPLACEMENT_AAS`.


In [ ]:
UBI_THRESHOLD = 0.40

MUTATION_MODE = "single"            # "single" or "combinatorial"
REPLACEMENT_AAS = ("A",)            # e.g. ("A", "R", "Q")
MAX_COMBINATION_ORDER = 2
MAX_VARIANTS = 5000

# S1 structural-preservation heuristic gates
GLOBAL_CA_RMSD_MAX = 1.0
LOCAL_MEAN_CA_DISPLACEMENT_MAX = 1.5
MUTATION_CA_DISPLACEMENT_MAX = 2.0
CONTACT_CHANGE_FRACTION_MAX = 0.10
MIN_MEAN_PLDDT = 70.0


## 4. User input


In [ ]:
PROTEIN_ID = "my_protein"
WT_SEQUENCE = """
PASTE_AMINO_ACID_SEQUENCE_HERE
"""


## 5. Build the workflow


In [ ]:
from therapeutic_optimization import (
    MutationConfig,
    PredictorConfig,
    StructuralThresholds,
    WorkflowConfig,
    OptimizationWorkflow,
)

config = WorkflowConfig(
    mutation=MutationConfig(
        threshold=UBI_THRESHOLD,
        mode=MUTATION_MODE,
        replacement_aas=REPLACEMENT_AAS,
        max_combination_order=MAX_COMBINATION_ORDER,
        max_variants=MAX_VARIANTS,
    ),
    ubiquitination=PredictorConfig(
        name="eup",
        threshold=UBI_THRESHOLD,
        eup_repo_dir=Path("/content/external/EUP"),
        model_cache_dir=Path("/content/huggingface"),
    ),
    structural_thresholds=StructuralThresholds(
        global_ca_rmsd_max=GLOBAL_CA_RMSD_MAX,
        local_mean_ca_displacement_max=LOCAL_MEAN_CA_DISPLACEMENT_MAX,
        mutation_ca_displacement_max=MUTATION_CA_DISPLACEMENT_MAX,
        contact_change_fraction_max=CONTACT_CHANGE_FRACTION_MAX,
        min_mean_plddt=MIN_MEAN_PLDDT,
    ),
)

workflow = OptimizationWorkflow(RUN_ROOT, config)


## T1 — input → WT FASTA


In [ ]:
t1 = workflow.T1(WT_SEQUENCE, PROTEIN_ID)
t1


## UP1 — WT ubiquitination prediction


In [ ]:
up1 = workflow.UP1()
display(up1)


## T2 — generate mutant FASTAs


In [ ]:
t2 = workflow.T2(up1)
print(f"Generated {len(t2)} mutant(s).")
display(t2)


## S1 — structure prediction + preservation screen

This is the expensive structure stage. Every T2 mutant is predicted and compared with WT.


In [ ]:
s1_metrics, s1_conserved = workflow.S1(t2, predict_structures=True)
print(f"Structurally conserved: {len(s1_conserved)} / {len(s1_metrics)}")
display(s1_metrics)
display(s1_conserved)


## R1 — record structural dropouts


In [ ]:
r1 = workflow.R1(t2, s1_metrics)
display(r1)


## UB2 — rerun ubiquitination prediction on S1 survivors


In [ ]:
ub2 = workflow.UB2(s1_conserved)
display(ub2)


## R2 — final ranking


In [ ]:
r2_all, optimized, needs_more = workflow.R2(up1, ub2, s1_conserved)

print("OPTIMIZED")
display(optimized)

print("NEEDS FURTHER OPTIMIZATION")
display(needs_more)

print("ALL RANKED")
display(r2_all)


## 6. Inspect outputs


In [ ]:
print("Tables:")
for path in sorted((RUN_ROOT / "storage" / "tables").glob("*.csv")):
    print(" -", path)

print("
Mutant FASTAs:", len(list((RUN_ROOT / "storage" / "mutants" / "fastas").glob("*.fasta"))))
print("WT structure directory:", RUN_ROOT / "storage" / "structures" / "wt")
print("Mutant structure directory:", RUN_ROOT / "storage" / "structures" / "mutants")
